In [0]:
from pyspark.sql.functions import col, count, explode, row_number, split, initcap, trim
from pyspark.sql.window import Window
from delta.tables import DeltaTable

SILVER_BOOKS_PATH = "s3://s3-goodreads-data/Silver/books"
SILVER_INTERACTIONS_PATH = "s3://s3-goodreads-data/Silver/interactions"
GOLD_BOOKS_PATH   = "s3://s3-goodreads-data/Gold/books"

# 1. READ FILTERED - NEW SILVER DATA
df_new_books = spark.read.format("delta").load(SILVER_BOOKS_PATH)
df_new_interactions = spark.read.format("delta").load(SILVER_INTERACTIONS_PATH)

# 2. AGGREGATE NEW INTERACTIONS
df_new_counts = df_new_interactions.groupBy("book_id") \
    .agg(count("*").alias("interaction_count"))

# 3. EXTRACT MAIN GENRE
df_exploded = df_new_books.select("book_id", explode("genres").alias("genre_name", "genre_count"))
window_spec = Window.partitionBy("book_id").orderBy(col("genre_count").desc())

df_new_main_genre = df_exploded.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .select(
        "book_id", 
        initcap(trim(split(col("genre_name"), ",")[0])).alias("main_genre")
    )

# 4. JOIN UPDATES
df_updates = df_new_books \
    .join(df_new_counts, on="book_id", how="left") \
    .join(df_new_main_genre, on="book_id", how="left") \
    .fillna({
        "interaction_count": 0,
        "main_genre": "Uncategorized"
    })

# 5. MERGE INTO GOLD
df_updates.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_BOOKS_PATH)

In [0]:
from pyspark.sql.functions import year, month, expr, count, col

spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")

INTERACTIONS_PATH = "s3://s3-goodreads-data/Silver/interactions"
GOLD_TRENDS_PATH  = "s3://s3-goodreads-data/Gold/monthly_trends"

df = spark.read.format("delta").load(INTERACTIONS_PATH)

df_trends = df.withColumn("interaction_date", expr("try_to_timestamp(date_added, 'E MMM dd HH:mm:ss Z yyyy')")) \
    .filter(col("interaction_date").isNotNull()) \
    .filter(year("interaction_date") > 2000) \
    .groupBy(year("interaction_date").alias("year"), month("interaction_date").alias("month")) \
    .agg(count("*").alias("total_interactions")) \
    .orderBy("year", "month")

df_trends.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_TRENDS_PATH)

In [0]:
from pyspark.sql.functions import col, explode, count, row_number
from pyspark.sql.window import Window

# 1. SETUP
SILVER_BOOKS   = "s3://s3-goodreads-data/Silver/books"
SILVER_AUTHORS = "s3://s3-goodreads-data/Silver/authors"
GOLD_AUTHORS   = "s3://s3-goodreads-data/Gold/authors"

df_books = spark.read.format("delta").load(SILVER_BOOKS)
df_authors = spark.read.format("delta").load(SILVER_AUTHORS)

# 2. CALCULATE TOP GENRE PER AUTHOR
# Step A: Explode Books to get Author IDs
df_book_authors = df_books.select(explode("authors.author_id").alias("author_id"), "genres")

# Step B: Explode Genres (Map -> Rows) to count them
df_author_genres = df_book_authors.select("author_id", explode("genres").alias("genre_name", "count")) \
    .groupBy("author_id", "genre_name") \
    .agg(count("*").alias("genre_frequency"))

# Step C: Pick the #1 Genre
window_spec = Window.partitionBy("author_id").orderBy(col("genre_frequency").desc())

df_top_genre = df_author_genres.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .select("author_id", col("genre_name").alias("main_genre"))

# 3. JOIN BACK TO AUTHORS - joining Silver Authors + Genre Data
df_final = df_authors.join(df_top_genre, on="author_id", how="left") \
    .fillna({"main_genre": "General"}) 

# 4. WRITE TO GOLD
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_AUTHORS)